<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_08_2_keras_ensembles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# T81-558: Anwendungen tiefer neuronaler Netzwerke
**Modul 8: Kaggle-Datensätze**
* Dozent: [Jeff Heaton](https://sites.wustl.edu/jeffheaton/), McKelvey School of Engineering, [Jeff Heaton](https://sites.wustl.edu/jeffheaton/)
* Weitere Informationen finden Sie unter [class website](https://sites.wustl.edu/jeffheaton/t81-558/).

# Modul 8 Material

* Teil 8.1: Einführung in Kaggle [[Video]](https://www.youtube.com/watch?v=7Mk46fb0Ayg&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=7Mk46fb0Ayg&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* **Teil 8.2: Erstellen von Ensembles mit Scikit-Learn und PyTorch** [[Video]](https://www.youtube.com/watch?v=przbLRCRL24&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=przbLRCRL24&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 8.3: Wie sollten Sie Ihr PyTorch-Neuralnetzwerk strukturieren: Hyperparameter [[Video]](https://www.youtube.com/watch?v=YTL2BR4U2Ng&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=YTL2BR4U2Ng&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 8.4: Bayesianische Hyperparameteroptimierung für PyTorch [[Video]](https://www.youtube.com/watch?v=1f4psgAcefU&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN) [[Video]](https://www.youtube.com/watch?v=1f4psgAcefU&list=PLjy4p-07OYzulelvJ5KVaT2pDlxivl_BN)
* Teil 8.5: Kaggle des aktuellen Semesters [[Video]] [[Notebook]](t81_558_class_08_5_kaggle_project.ipynb)

# Google CoLab-Anweisungen

Der folgende Code stellt sicher, dass Google CoLab die richtige Version von TensorFlow ausführt.
Durch Ausführen des folgenden Codes wird Ihr GDrive ```/content/drive``` zugeordnet.

In [1]:
try:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False


# Schön formatierte Zeitzeichenfolge
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)


# Frühzeitiges Stoppen (siehe Modul 3.4)
import copy


class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_model = None
        self.best_loss = None
        self.counter = 0
        self.status = ""

    def __call__(self, model, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_model = copy.deepcopy(model.state_dict())
        elif self.best_loss - val_loss >= self.min_delta:
            self.best_model = copy.deepcopy(model.state_dict())
            self.best_loss = val_loss
            self.counter = 0
            self.status = f"Improvement found, counter reset to {self.counter}"
        else:
            self.counter += 1
            self.status = f"No improvement in the last {self.counter} epochs"
            if self.counter >= self.patience:
                self.status = f"Early stopping triggered after {self.counter} epochs."
                if self.restore_best_weights:
                    model.load_state_dict(self.best_model)
                return True
        return False


# Nutzen Sie eine GPU oder MPS (Apple), sofern verfügbar. (siehe Modul 3.2)
import torch

has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Verwendetes Gerät: {device}")

Note: not using Google CoLab
Using device: mps


# Teil 8.2: Erstellen von Ensembles mit Scikit-Learn und PyTorch

### Bewertung der Feature-Wichtigkeit

Die Merkmalswichtigkeit gibt an, wie wichtig jedes Merkmal (aus dem Merkmals-/Importvektor) für die Vorhersage eines neuronalen Netzwerks oder eines anderen Modells ist. Es gibt viele verschiedene Möglichkeiten, die Merkmalswichtigkeit neuronaler Netzwerke zu bewerten. Das folgende Dokument bietet einen hervorragenden (und gut lesbaren) Überblick über die verschiedenen Möglichkeiten zur Bewertung der Bedeutung von Eingaben/Merkmalen neuronaler Netzwerke.

* Ein genauer Vergleich von Methoden zur Quantifizierung der Variablenbedeutung in künstlichen neuronalen Netzwerken unter Verwendung simulierter Daten [[Cite:olden2004accurate]](http://depts.washington.edu/oldenlab/wordpress/wp-content/uploads/2013/03/EcologicalModelling_2004.pdf). *Ecological Modelling*, 178(3), 389-397.

Zusammenfassend stehen neuronalen Netzen folgende Methoden zur Verfügung:

* Verbindungsgewichte-Algorithmus
* Partielle Ableitungen
* Eingangsstörung
* Sensitivitätsanalyse
* Vorwärts-Schrittweise-Addition
* Verbesserte schrittweise Auswahl 1
* Rückwärtsgerichtete schrittweise Eliminierung
* Verbesserte schrittweise Auswahl

In diesem Kapitel verwenden wir den Eingabe-Perturbation-Feature-Ranking-Algorithmus. Dieser Algorithmus funktioniert mit jedem Regressions- oder Klassifizierungsnetzwerk. Im nächsten Abschnitt stelle ich eine Implementierung des Eingabe-Perturbation-Algorithmus für scikit-learn bereit. Dieser Code implementiert eine Funktion weiter unten, die mit jedem scikit-learn-Modell funktioniert.

[Leo Breiman](https://en.wikipedia.org/wiki/Leo_Breiman) hat diesen Algorithmus in seinem wegweisenden Artikel über Random Forests vorgestellt. [Leo Breiman](https://en.wikipedia.org/wiki/Leo_Breiman) Obwohl er diesen Algorithmus in Verbindung mit Random Forests vorgestellt hat, ist er modellunabhängig und für jedes überwachte Lernmodell geeignet. Dieser Algorithmus, bekannt als Input-Perturbation-Algorithmus, funktioniert, indem er die Genauigkeit eines trainierten Modells bewertet, wobei jeder Input einzeln aus einem Datensatz gemischt wird. Durch das Mischen eines Inputs wird dieser nutzlos – er wird effektiv aus dem Modell entfernt. Wichtigere Inputs führen zu einem weniger genauen Ergebnis, wenn sie durch Mischen entfernt werden. Dieser Prozess ist sinnvoll, da wichtige Merkmale zur Genauigkeit des Modells beitragen. Ich habe die TensorFlow-Implementierung dieses Algorithmus erstmals im folgenden Artikel vorgestellt.

* Bedeutung der frühen Stabilisierungsfunktion für tiefe neuronale Netzwerke von TensorFlow[[Cite:heaton2017early]](https://www.heatonresearch.com/dload/phd/IJCNN%202017-v2-final.pdf)

Dieser Algorithmus nutzt den Log-Verlust zur Bewertung eines Klassifizierungsproblems und RMSE zur Regression.

In [2]:
from sklearn import metrics
import scipy as sp
import numpy as np
import math
from sklearn import metrics

import torch
import torch.nn.functional as F
import pandas as pd


def perturbation_rank(device, model, x, y, names, regression):
    model.to(device)
    model.eval()  # Setzen Sie das Modell in den Auswertungsmodus

    # x = torch.tensor(x).float().to(Gerät)
    # y = torch.tensor(y).float().to(Gerät)

    errors = []

    for i in range(x.shape[1]):
        hold = x[:, i].clone()
        x[:, i] = torch.randperm(x.shape[0]).to(device)  # schlurfend

        with torch.no_grad():
            pred = model(x)

        if regression:
            loss_fn = torch.nn.MSELoss()
            error = loss_fn(y, pred).item()
        else:
            # pred sollte Wahrscheinlichkeiten sein; wenden Sie Softmax an, wenn dies nicht in der Vorwärtsmethode des Modells erfolgt
            if len(pred.shape) == 2 and pred.shape[1] > 1:
                pred = F.softmax(pred, dim=1)
                loss_fn = torch.nn.CrossEntropyLoss()
                error = loss_fn(pred, y.long()).item()
            else:
                loss_fn = nn.MSELoss()
                error = loss_fn(y, pred).item()

        errors.append(error)
        x[:, i] = hold

    max_error = max(errors)
    importance = [e / max_error for e in errors]

    data = {"name": names, "error": errors, "importance": importance}
    result = DataFrame(data, columns=["name", "error", "importance"])
    result.sort_values(by=["importance"], ascending=[0], inplace=True)
    result.reset_index(inplace=True, drop=True)
    return result

## Klassifizierung und Rangfolge der Eingangsstörungen

Wir sehen uns nun den Code zur Durchführung der Störungsbewertung für ein neuronales Klassifizierungsnetzwerk an. Die Implementierungstechnik unterscheidet sich für Klassifizierung und Regression geringfügig, daher muss ich zwei verschiedene Implementierungen bereitstellen. Der Hauptunterschied zwischen Klassifizierung und Regression besteht darin, wie wir die Genauigkeit des neuronalen Netzwerks in jedem dieser beiden Netzwerktypen bewerten. Wir verwenden die Fehlerberechnung Root Mean Square (RMSE), während wir für die Klassifizierung den Log-Loss verwenden.

Der unten dargestellte Code erstellt ein Klassifizierungs-Neuronales Netzwerk, das den klassischen Iris-Datensatz vorhersagt.

In [3]:
# Ausgabe ausblenden
import time

import numpy as np
import pandas as pd
import torch
import tqdm
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Labelcoder, StandardScaler
from torch import nn
from torch.autograd import Variable
from torch.utils.data import DataLoader, TensorDataset

# Legen Sie einen Zufallsstartwert für die Reproduzierbarkeit fest
np.random.seed(42)
torch.manual_seed(42)


def load_data():
    df = read_csv(
        "https://data.heatonresearch.com/data/t81-558/iris.csv", na_values=["NA", "?"]
    )

    le = Labelcoder()

    x = df[["sepal_l", "sepal_w", "petal_l", "petal_w"]].values
    y = le.fit_transform(df["species"])
    species = le.classes_

    # Aufteilung in Validierungs- und Trainingssets
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.25, random_state=42
    )

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)

    # Numpy zu Torch Tensor
    x_train = torch.tensor(x_train, device=device, dtype=torch.float32)
    y_train = torch.tensor(y_train, device=device, dtype=torch.long)

    x_test = torch.tensor(x_test, device=device, dtype=torch.float32)
    y_test = torch.tensor(y_test, device=device, dtype=torch.long)

    return x_train, x_test, y_train, y_test, species, df.columns


x_train, x_test, y_train, y_test, species, columns = load_data()
columns = list(columns)
columns.remove("species")  # entferne das Ziel(y)

# Erstellen von Datasets
BATCH_SIZE = 16

dataset_train = TensorDataset(x_train, y_train)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)

dataset_test = TensorDataset(x_test, y_test)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=True)

# Erstellen Sie ein Modell mit nn.Sequential
model = nn.Sequential(
    nn.Linear(x_train.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, len(species)),
    nn.LogSoftmax(dim=1),
)

model = torch.compile(model, backend="aot_eager").to(device)

loss_fn = nn.CrossEntropyLoss()  # Kreuzentropieverlust

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
es = EarlyStopping()

epoch = 0
done = False
while epoch < 1000 and not done:
    epoch += 1
    steps = list(enumerate(dataloader_train))
    pbar = tqdm.tqdm(steps)
    model.train()
    for i, (x_batch, y_batch) in pbar:
        y_batch_pred = model(x_batch.to(device))
        loss = loss_fn(y_batch_pred, y_batch.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss, current = loss.item(), (i + 1) * len(x_batch)
        if i == len(steps) - 1:
            model.eval()
            pred = model(x_test)
            vloss = loss_fn(pred, y_test)
            if es(model, vloss):
                done = True
            pbar.set_description(
                f"Epoch: {epoch}, tloss: {loss}, vloss: {vloss:>7f}, {es.status}"
            )
        else:
            pbar.set_description(f"Epoch: {epoch}, tloss {loss:}")

Epoch: 1, tloss: 0.6026307344436646, vloss: 0.536555, : 100%|██████████| 7/7 [00:00<00:00, 14.55it/s]
Epoch: 2, tloss: 0.36586475372314453, vloss: 0.277725, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 172.42it/s]
Epoch: 3, tloss: 0.15603026747703552, vloss: 0.187535, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 133.04it/s]
Epoch: 4, tloss: 0.05794892832636833, vloss: 0.154333, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 229.42it/s]
Epoch: 5, tloss: 0.18528980016708374, vloss: 0.076723, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 240.43it/s]
Epoch: 6, tloss: 0.12420052289962769, vloss: 0.061499, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 237.66it/s]
Epoch: 7, tloss: 0.0334041602909565, vloss: 0.045322, Improvement found, counter reset to 0: 100%|██████████| 7/7 [00:00<00:00, 226.48it/s]
Epoch: 8, tloss: 0.09452516585588455, vloss: 0.032975

Als Nächstes bewerten wir die Genauigkeit des trainierten Modells. Hier sehen wir, dass das neuronale Netzwerk mit einer Genauigkeit von 1,0 eine hervorragende Leistung erbringt. Bei einem komplexeren Datensatz mit einer so hohen Genauigkeit befürchten wir möglicherweise eine Überanpassung. In diesem Beispiel sind wir jedoch eher daran interessiert, die Wichtigkeit jeder Spalte zu bestimmen.

In [4]:
from sklearn.metrics import accuracy_score

pred = model(x_test)
_, predict_classes = torch.max(pred, 1)
correct = accuracy_score(y_test.cpu(), predict_classes.cpu())
print(f"Genauigkeit: {korrekt}")

Accuracy: 1.0


Wir sind nun bereit, den Eingabestörungsalgorithmus aufzurufen. Zuerst extrahieren wir die Spaltennamen und entfernen die Zielspalte. Die Zielspalte ist nicht wichtig, da sie das Ziel und nicht eine der Eingaben ist. Beim überwachten Lernen ist das Ziel von größter Bedeutung.

Wir können die Wichtigkeit in der folgenden Tabelle sehen. Die wichtigste Spalte ist immer 1,0 und die weniger wichtigen Spalten weisen weiterhin einen Abwärtstrend auf. Die am wenigsten wichtige Spalte hat den niedrigsten Rang.

In [5]:
# Bewerten Sie die Funktionen
from IPython.display import display, HTML

rank = perturbation_rank(device, model, x_test, y_test, columns, False)
display(rank)

,name,error,importance
0,petal_w,1.229601,1.000000
1,petal_l,1.228287,0.998932
2,sepal_w,1.155053,0.939373
3,sepal_l,0.976901,0.794486


## Regression und Rangfolge der Eingangsstörungen

Wir sehen nun, wie man die Rangfolge der Eingangsstörungen für ein neuronales Regressionsnetzwerk verwendet. Wir werden den MPG-Datensatz als Demonstration verwenden. Der folgende Code lädt den MPG-Datensatz und erstellt ein neuronales Regressionsnetzwerk für diesen Datensatz. Der Code trainiert das neuronale Netzwerk und berechnet eine RMSE-Bewertung.

In [6]:
# Ausgabe ausblenden
import time

import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import tqdm
from sklearn import preprocessing
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.autograd import Variable
from torch.utils.data import DataLoader, TensorDataset

# Lesen Sie den MPG-Datensatz.
df = read_csv(
    "https://data.heatonresearch.com/data/t81-558/auto-mpg.csv", na_values=["NA", "?"]
)

cars = df["name"]

# Fehlende Werte behandeln
df["horsepower"] = df["horsepower"].fillna(df["horsepower"].median())

# Pandas zu Numpy
x = df[
    [
        "cylinders",
        "displacement",
        "horsepower",
        "weight",
        "acceleration",
        "year",
        "origin",
    ]
].values
y = df["mpg"].values  # Rückschritt

# Aufteilung in Validierungs- und Trainingssets
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42
)

# Numpy zu Torch Tensor
x_train = torch.tensor(x_train, device=device, dtype=torch.float32)
y_train = torch.tensor(y_train, device=device, dtype=torch.float32)

x_test = torch.tensor(x_test, device=device, dtype=torch.float32)
y_test = torch.tensor(y_test, device=device, dtype=torch.float32)


# Erstellen von Datasets
BATCH_SIZE = 16

dataset_train = TensorDataset(x_train, y_train)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)

dataset_test = TensorDataset(x_test, y_test)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=True)


# Modell erstellen

model = nn.Sequential(
    nn.Linear(x_train.shape[1], 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1),
)

model = torch.compile(model, backend="aot_eager").to(device)

# Definieren Sie die Verlustfunktion für die Regression
loss_fn = nn.MSELoss()

# Definieren Sie den Optimierer
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

es = EarlyStopping()

epoch = 0
done = False
while epoch < 1000 and not done:
    epoch += 1
    steps = list(enumerate(dataloader_train))
    pbar = tqdm.tqdm(steps)
    model.train()
    for i, (x_batch, y_batch) in pbar:
        y_batch_pred = model(x_batch).flatten()  #
        loss = loss_fn(y_batch_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss, current = loss.item(), (i + 1) * len(x_batch)
        if i == len(steps) - 1:
            model.eval()
            pred = model(x_test).flatten()
            vloss = loss_fn(pred, y_test)
            if es(model, vloss):
                done = True
            pbar.set_description(
                f"Epoch: {epoch}, tloss: {loss}, vloss: {vloss:>7f}, EStop:[{es.status}]"
            )
        else:
            pbar.set_description(f"Epoch: {epoch}, tloss {loss:}")

from sklearn import metrics

# Messen Sie den RMSE-Fehler. RMSE wird häufig bei Regressionen eingesetzt.
pred = model(x_test)
score = torch.sqrt(torch.nn.functional.mse_loss(pred.flatten(), y_test))
print(f"Endergebnis (RMSE): {score}")

Epoch: 1, tloss: 224.4937286376953, vloss: 275.311096, EStop:[]: 100%|██████████| 19/19 [00:00<00:00, 63.33it/s]
Epoch: 2, tloss: 221.47691345214844, vloss: 186.099442, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<00:00, 248.06it/s]
Epoch: 3, tloss: 238.82725524902344, vloss: 150.277847, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<00:00, 255.10it/s]
Epoch: 4, tloss: 120.80052947998047, vloss: 131.800980, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<00:00, 196.25it/s]
Epoch: 5, tloss: 134.80111694335938, vloss: 154.462509, EStop:[No improvement in the last 1 epochs]: 100%|██████████| 19/19 [00:00<00:00, 242.83it/s]
Epoch: 6, tloss: 88.8158187866211, vloss: 101.267807, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<00:00, 234.08it/s]
Epoch: 7, tloss: 54.8061408996582, vloss: 73.606964, EStop:[Improvement found, counter reset to 0]: 100%|██████████| 19/19 [00:00<0

Final score (RMSE): 3.7894699573516846


Genau wie zuvor extrahieren wir die Spaltennamen und verwerfen das Ziel. Wir können jetzt eine Rangfolge der Wichtigkeit der einzelnen Eingabefunktionen erstellen. Die Funktion mit der Rangfolge 1,0 ist die wichtigste.

In [7]:
# Bewerten Sie die Funktionen
from IPython.display import display, HTML

names = list(df.columns)  # x+y-Spaltennamen
names.remove("name")
names.remove("mpg")  # entferne das Ziel(y)
rank = perturbation_rank(device, model, x_test, y_test, names, True)
display(rank)

/Users/jeff/miniconda3/envs/torch/lib/python3.9/site-packages/torch/nn/modules/loss.py:536: UserWarning: Using a target size (torch.Size([100, 1])) that is different to the input size (torch.Size([100])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


,name,error,importance
0,origin,718.869507,1.000000
1,weight,376.961060,0.524380
2,year,278.980316,0.388082
3,acceleration,208.227646,0.289660
4,displacement,192.715042,0.268081
5,horsepower,128.381210,0.178588
6,cylinders,120.705498,0.167910


## Biologische Reaktion mit neuronalem Netzwerk

In den folgenden Abschnitten wird gezeigt, wie Sie die Rangfolge der Merkmalswichtigkeit und das Ensembleing mit einem komplexeren Datensatz verwenden. Ensembleing ist der Prozess, bei dem Sie mehrere Modelle kombinieren, um eine höhere Genauigkeit zu erzielen. Gewinner von Kaggle-Wettbewerben verwenden häufig Ensembleing für hochrangige Lösungen.

Wir verwenden den Datensatz zur biologischen Reaktion, einen Kaggle-Datensatz, der eine ungewöhnlich hohe Anzahl von Spalten aufweist. Aufgrund der großen Anzahl von Spalten ist es wichtig, die Merkmalsrangfolge zu verwenden, um die Wichtigkeit dieser Spalten zu bestimmen. Wir beginnen mit dem Laden des Datensatzes und der Vorverarbeitung. Dieser Kaggle-Datensatz ist ein binäres Klassifizierungsproblem. Sie müssen vorhersagen, ob bestimmte Bedingungen eine biologische Reaktion auslösen.

* [Predicting a Biological Response](https://www.kaggle.com/c/bioresponse)

In [8]:
import pandas as pd
import numpy as np
from sklearn import metrics
from scipy.stats import zscore
from sklearn.model_selection import KFold
from IPython.display import HTML, display

URL = "https://data.heatonresearch.com/data/t81-558/kaggle/"

df_train = read_csv(URL + "bio_train.csv", na_values=["NA", "?"])

df_test = read_csv(URL + "bio_test.csv", na_values=["NA", "?"])

activity_classes = df_train["Activity"]

Eine große Anzahl von Spalten wird deutlich, wenn wir die Form des Datensatzes anzeigen.

In [9]:
print(df_train.shape)

(3751, 1777)


Der folgende Code erstellt ein neuronales Klassifizierungsnetzwerk und trainiert es für den Datensatz biologischer Reaktionen. Nach dem Training wird die Genauigkeit gemessen.

In [10]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import numpy as np
import sklearn
from sklearn import metrics
from torch.utils.data import DataLoader, TensorDataset

# Vorausgesetzt, df_train und df_test sind vordefiniert
x_columns = df_train.columns.drop("Activity")
x = torch.tensor(df_train[x_columns].values, dtype=torch.float32)
y = torch.tensor(df_train["Activity"].values, dtype=torch.float32).view(
    -1, 1
)  # Für binäre Kreuzentropie
x_submit = torch.tensor(df_test[x_columns].values, dtype=torch.float32)

# Aufteilung in Trainieren/Testen
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42
)

# Zur GPU wechseln, falls verfügbar
x_train, y_train, x_test, y_test = map(
    lambda t: t.clone().detach().to(device), (x_train, y_train, x_test, y_test)
)

train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Definieren Sie das Modell mit Sequential
model = nn.Sequential(
    nn.Linear(x_train.shape[1], 25),
    nn.ReLU(),
    nn.Linear(25, 10),
    nn.Linear(10, 1),
    nn.Sigmoid(),
).to(device)

# Verlust und Optimierer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training mit frühzeitigem Abbruch
best_loss = float("inf")
patience = 5
no_improvement = 0

for epoch in range(1000):
    model.train()
    for batch in train_loader:
        inputs, labels = batch

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = sum(
            criterion(model(inputs), labels) for inputs, labels in test_loader
        )
        if val_loss < best_loss - 1e-3:
            best_loss = val_loss
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= patience:
            print("Early stopping")
            break

# Vorhersage
with torch.no_grad():
    pred = model(x_test).cpu().numpy().flatten()
    pred = np.clip(pred, a_min=1e-6, a_max=1 - 1e-6)

    print("Validation logloss: {}".format(sklearn.metrics.log_loss(y_test.cpu(), pred)))

    pred_binary = (pred > 0.5).astype(int)
    score = metrics.accuracy_score(y_test.cpu().numpy(), pred_binary)
    print("Validation accuracy score: {}".format(score))

    pred_submit = model(x_submit.to(device)).cpu().numpy().flatten()
    pred_submit = np.clip(pred_submit, a_min=1e-6, a_max=1 - 1e-6)

    submit_df = DataFrame(
        {
            "MoleculeId": [x + 1 for x in range(len(pred_submit))],
            "PredictedProbability": pred_submit,
        }
    )
    submit_df.to_csv("submit.csv", index=False)

Early stopping
Validation logloss: 0.5384844082050355
Validation accuracy score: 0.7750533049040512


## Welche Funktionen/Spalten sind wichtig
Im Folgenden wird zur Bewertung des neuronalen Netzwerks eine Störungsrangfolge verwendet.

In [11]:
# Bewerten Sie die Funktionen
from IPython.display import display, HTML

names = list(df_train.columns)  # x+y-Spaltennamen
names.remove("Activity")  # entferne das Ziel(y)
rank = perturbation_rank(device, model, x_test, y_test, names, False)
display(rank[0:10])

,name,error,importance
0,D860,0.570163,1.000000
1,D288,0.570031,0.999768
2,D663,0.569986,0.999690
3,D671,0.569706,0.999198
4,D101,0.569513,0.998860
5,D1576,0.569312,0.998508
6,D206,0.569083,0.998106
7,D655,0.568983,0.997929
8,D1480,0.568912,0.997806
9,D179,0.568880,0.997750


## Neuronales Netzwerk-Ensemble

Ein Ensemble neuronaler Netze kombiniert Vorhersagen neuronaler Netze mit anderen Modellen. Das Programm ermittelt die genaue Mischung dieser Modelle durch logistische Regression. Der folgende Code führt diese Mischung für eine Klassifizierung durch. Wenn Sie Kaggle die endgültigen Vorhersagen des Ensembles präsentieren, werden Sie sehen, dass das Ergebnis sehr genau ist.

In [12]:
import numpy as np
import os
import pandas as pd
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

SHUFFLE = False
FOLDS = 10


# Verwenden von nn.Sequential zum Definieren des Modells
def build_ann(input_size, classes, neurons):
    model = nn.Sequential(
        nn.Linear(input_size, neurons),
        nn.ReLU(),
        nn.Linear(neurons, 1),
        nn.Linear(1, classes),
        nn.Softmax(dim=1),
    )
    return model


def mlogloss(y_test, preds):
    epsilon = 1e-15
    sum = 0
    for row in zip(preds, y_test):
        x = row[0][row[1]]
        x = max(epsilon, x)
        x = min(1 - epsilon, x)
        sum += math.log(x)
    return (-1 / len(preds)) * sum


def stretch(y):
    return (y - y.min()) / (y.max() - y.min())


def blend_ensemble(x, y, x_submit):
    kf = StratifiedKFold(FOLDS)
    folds = list(kf.split(x, y))

    models = [
        build_ann(x.shape[1], 2, 20),
        KNeighborsClassifier(n_neighbors=3),
        RandomForestClassifier(n_estimators=100, n_jobs=-1, criterion="gini"),
        RandomForestClassifier(n_estimators=100, n_jobs=-1, criterion="entropy"),
        ExtraTreesClassifier(n_estimators=100, n_jobs=-1, criterion="gini"),
        ExtraTreesClassifier(n_estimators=100, n_jobs=-1, criterion="entropy"),
        GradientBoostingClassifier(
            learning_rate=0.05, subsample=0.5, max_depth=6, n_estimators=50
        ),
    ]

    dataset_blend_train = np.zeros((x.shape[0], len(models)))
    dataset_blend_test = np.zeros((x_submit.shape[0], len(models)))

    for j, model in enumerate(models):
        print("Model: {} : {}".format(j, model))
        fold_sums = np.zeros((x_submit.shape[0], len(folds)))
        total_loss = 0
        for i, (train, test) in enumerate(folds):
            x_train = torch.tensor(x[train], dtype=torch.float32)
            y_train = torch.tensor(y[train].values, dtype=torch.int64)
            x_test = torch.tensor(x[test], dtype=torch.float32)
            y_test = torch.tensor(y[test].values, dtype=torch.int64)

            if isinstance(
                model, nn.Module
            ):  # Überprüfen Sie, ob das Modell ein PyTorch-Modell ist
                optimizer = optim.Adam(model.parameters())
                criterion = nn.CrossEntropyLoss()

                # Ausbildung
                optimizer.zero_grad()
                outputs = model(x_train)
                loss = criterion(outputs, y_train)
                loss.backward()
                optimizer.step()

                # Vorhersage
                with torch.no_grad():
                    outputs_test = model(x_test)
                    _, predicted = outputs_test.max(1)
                    pred = F.softmax(outputs_test, dim=1).numpy()
                    outputs_submit = model(torch.tensor(x_submit, dtype=torch.float32))
                    pred2 = F.softmax(outputs_submit, dim=1).numpy()
            else:
                model.fit(x_train, y_train)
                pred = np.array(model.predict_proba(x_test))
                pred2 = np.array(model.predict_proba(x_submit))

            dataset_blend_train[test, j] = pred[:, 1]
            fold_sums[:, i] = pred2[:, 1]
            loss = mlogloss(y_test, pred)
            total_loss += loss
            print("Fold # {}: Verlust={}".format(i, Verlust))
        print(
            "{}: Mean loss={}".format(model.__class__.__name__, total_loss / len(folds))
        )
        dataset_blend_test[:, j] = fold_sums.mean(1)

    print()
    print("Blending models.")
    blend = LogisticRegression(solver="lbfgs")
    blend.fit(dataset_blend_train, y)
    return blend.predict_proba(dataset_blend_test)


if __name__ == "__main__":
    np.random.seed(42)  # Seed zum Mischen des Eisenbahnsets

    print("Loading data...")
    URL = "https://data.heatonresearch.com/data/t81-558/kaggle/"

    df_train = read_csv(URL + "bio_train.csv", na_values=["NA", "?"])
    df_submit = read_csv(URL + "bio_test.csv", na_values=["NA", "?"])

    predictors = list(df_train.columns.values)
    predictors.remove("Activity")
    x = df_train[predictors].values
    y = df_train["Activity"]
    x_submit = df_submit.values

    if SHUFFLE:
        idx = np.random.permutation(y.size)
        x = x[idx]
        y = y[idx]

    submit_data = blend_ensemble(x, y, x_submit)
    submit_data = stretch(submit_data)

    # Submit-Datei erstellen
    ids = [id + 1 for id in range(submit_data.shape[0])]
    submit_df = DataFrame(
        {"MoleculeId": ids, "PredictedProbability": submit_data[:, 1]},
        columns=["MoleculeId", "PredictedProbability"],
    )
    submit_df.to_csv("submit.csv", index=False)

Loading data...
Model: 0 : Sequential(
  (0): Linear(in_features=1776, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=1, bias=True)
  (3): Linear(in_features=1, out_features=2, bias=True)
  (4): Softmax(dim=1)
)
Fold #0: loss=0.7723120821060854
Fold #1: loss=0.7621972254727901
Fold #2: loss=0.7543882373575242
Fold #3: loss=0.7438802241516417
Fold #4: loss=0.7359048127781634
Fold #5: loss=0.7294116013880749
Fold #6: loss=0.7212477197135376
Fold #7: loss=0.7070651846400796
Fold #8: loss=0.7078284621313083
Fold #9: loss=0.7054422140402271
Sequential: Mean loss=0.7339677763779433
Model: 1 : KNeighborsClassifier(n_neighbors=3)
Fold #0: loss=3.606678388314123
Fold #1: loss=2.2256421551487593
Fold #2: loss=3.6815437059542186
Fold #3: loss=2.416161292225968
Fold #4: loss=4.442472310149748
Fold #5: loss=4.321350530738247
Fold #6: loss=3.400455469543658
Fold #7: loss=3.1724147110842513
Fold #8: loss=2.117356283193681
Fold #9: loss=3.0532135963322586
KNeighbo